# NCTBench-Pilot — RAG Evaluation: 12 Configuration Benchmark
3 retrievers × 2 generators × 2 prompt types = 12 configurations.
This is the main results section of the paper.
Corresponds to Section 4 (Results) and Section 5 (Analysis).

In [ ]:
import sys, json, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, str(Path(r"E:\CSAA Project\pipeline")))
sys.path.insert(0, str(Path(r"E:\CSAA Project\evaluation")))

## The 12 Evaluation Configurations

In [ ]:
configs = [
    (1,"bm25","llama","grounded"),   (2,"bm25","llama","open"),
    (3,"bm25","mixtral","grounded"), (4,"bm25","mixtral","open"),
    (5,"dense","llama","grounded"),  (6,"dense","llama","open"),
    (7,"dense","mixtral","grounded"),(8,"dense","mixtral","open"),
    (9,"hybrid","llama","grounded"), (10,"hybrid","llama","open"),
    (11,"hybrid","mixtral","grounded"),(12,"hybrid","mixtral","open"),
]
print(f"{'ID':3s} {'Retriever':8s} {'Generator':8s} {'Prompt':10s}")
print("-" * 35)
for c in configs:
    print(f"{c[0]:3d} {c[1]:8s} {c[2]:8s} {c[3]:10s}")

## Initialize All Retrievers

In [ ]:
from retrievers import BM25Retriever, DenseRetriever, HybridRRFRetriever
import json
from pathlib import Path
chunks = json.loads(
    Path(r"E:\CSAA Project\chunks\all_chunks.json")
    .read_text(encoding="utf-8")
)
bm25   = BM25Retriever(chunks)
dense  = DenseRetriever()
hybrid = HybridRRFRetriever(bm25, dense)
print(f"BM25  : {len(chunks)} chunks indexed")
print(f"Dense : ChromaDB loaded")
print(f"Hybrid: RRF k=60 ready")

## Live Retrieval Demo — Compare All 3 Retrievers

In [ ]:
query = "What is ICT and why is it important for students?"
print(f"Query: {query}\n")
for name, r in [("BM25", bm25), ("Dense", dense), ("Hybrid", hybrid)]:
    results = r.retrieve(query, k=3)
    print(f"=== {name} ===")
    for i, chunk in enumerate(results):
        print(f"  {i+1}. [{chunk['chunk_id']}] "
              f"{chunk.get('source','?')} | "
              f"class{chunk.get('class_num','?')} | "
              f"{chunk.get('language','?')}")
        print(f"     {chunk['text'][:150]}...")
    print()

## Live Generation Demo — Grounded vs Open

In [ ]:
from generators import RAGGenerator
gen = RAGGenerator()
query = "What is the purpose of an operating system?"
chunks_for_demo = dense.retrieve(query, k=3)
grounded_ans = gen.generate_grounded(query, chunks_for_demo)
open_ans     = gen.generate_open(query)
print(f"Question: {query}\n")
print(f"GROUNDED ANSWER:\n{grounded_ans}\n")
print(f"OPEN ANSWER:\n{open_ans}")

## Run Full Evaluation (12 configs × 200 questions)

In [ ]:
import subprocess
result = subprocess.run(
    ["conda", "run", "-n", "torch_env", "python",
     str(Path(r"E:\CSAA Project\evaluation\run_evaluation.py"))],
    capture_output=True, text=True, cwd=r"E:\CSAA Project\evaluation"
)
output = result.stdout
if len(output) > 4000:
    print("... (truncated) ...\n")
    print(output[-4000:])
else:
    print(output)
if result.returncode != 0:
    print("ERRORS:", result.stderr[-2000:])

## Results Analysis

In [ ]:
df = pd.read_csv(
    Path(r"E:\CSAA Project\evaluation\results\summary_table.csv")
)
display_cols = ["config_id","retriever","generator",
                "prompt_type","recall_at_3","token_f1",
                "em","hallucination_rate"]
print(df[display_cols].to_string(index=False))
best = df.loc[df["token_f1"].idxmax()]
print(f"\nBest config by Token F1: Config {int(best['config_id'])}")
print(f"  {best['retriever']} | {best['generator']} | {best['prompt_type']}")
print(f"  Recall@3 = {best['recall_at_3']:.3f}")
print(f"  Token F1 = {best['token_f1']:.3f}")
print(f"  EM       = {best['em']:.3f}")
print(f"  Hall.Rate= {best['hallucination_rate']:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = {"bm25":"#0072B2","dense":"#E69F00","hybrid":"#009E73"}

# Retriever comparison
ret_means = df.groupby("retriever")["token_f1"].mean()
axes[0].bar(ret_means.index,
            ret_means.values,
            color=[colors[r] for r in ret_means.index])
axes[0].set_title("Token F1 by Retriever")
axes[0].set_ylabel("Mean Token F1")
axes[0].set_ylim(0, 1)
for i, (r, v) in enumerate(ret_means.items()):
    axes[0].text(i, v+0.01, f"{v:.3f}", ha="center", fontsize=10)

# Bengali vs English
bn_mean = df["bn_f1"].mean()
en_mean = df["en_f1"].mean()
axes[1].bar(["Bengali","English"], [bn_mean, en_mean],
            color=["#0072B2","#E69F00"])
axes[1].set_title("Bengali vs English Token F1")
axes[1].set_ylabel("Mean Token F1")
axes[1].set_ylim(0, 1)
for i, v in enumerate([bn_mean, en_mean]):
    axes[1].text(i, v+0.01, f"{v:.3f}", ha="center", fontsize=10)

# Grounded vs Open
grounded = df[df["prompt_type"]=="grounded"]["token_f1"].values
open_    = df[df["prompt_type"]=="open"]["token_f1"].values
x = range(min(len(grounded), len(open_)))
axes[2].plot(list(x), grounded[:len(x)],
             "b-o", label="Grounded", linewidth=2)
axes[2].plot(list(x), open_[:len(x)],
             "r--s", label="Open", linewidth=2)
axes[2].fill_between(list(x),
                     grounded[:len(x)],
                     open_[:len(x)],
                     alpha=0.15, color="blue")
axes[2].set_title("Grounded vs Open Prompt")
axes[2].set_xlabel("Config Index")
axes[2].set_ylabel("Token F1")
axes[2].legend()

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle("NCTBench-Pilot RAG Evaluation Results", fontsize=13)
plt.tight_layout()
plt.savefig(r"E:\CSAA Project\notebooks\rag_results.png",
            dpi=150, bbox_inches="tight")
plt.show()